# AC-Solver Iterative Refinement — Colab Runner

**Runtime recommendation: High-RAM CPU**  
The search (V-guided greedy, beam) is CPU-only. GPU only helps training, which takes ~10 min per iteration — not the bottleneck.

**Normal workflow:**
- **First time / clean restart:** Cells 1 → 2 → 3 → 4 → 5 → 6
- **After Colab disconnect:** Cells 1 → 2 → 3 → 5 → 8 *(state persists on Drive)*
- **If state files are missing but partial results exist:** Cells 1 → 2 → 3 → 5 → 7 → 8

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


## Cell 2 — Clone or Update Repo on Drive

**First time only:** enter your GitHub Personal Access Token when prompted.  
Get one at: GitHub → Settings → Developer Settings → Personal access tokens → Fine-grained → repo read access.  
After first clone, this cell just does `git pull`.

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/Avi161/AC-Solver-Caltech.git'
DRIVE_DIR = '/content/drive/MyDrive/AC-Solver-Caltech'
BRANCH = 'feat/test-new-exp-claude'

if not os.path.exists(DRIVE_DIR):
    # First time: clone with token
    from getpass import getpass
    token = getpass('GitHub Personal Access Token: ')
    auth_url = REPO_URL.replace('https://', f'https://{token}@')
    subprocess.run(['git', 'clone', auth_url, DRIVE_DIR], check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=DRIVE_DIR, check=True)
    print('Repo cloned to Drive.')
else:
    # Already cloned: checkout branch first, then pull
    subprocess.run(['git', 'checkout', BRANCH], cwd=DRIVE_DIR, check=True)
    result = subprocess.run(['git', 'pull'], cwd=DRIVE_DIR, capture_output=True, text=True)
    print(result.stdout or 'Already up to date.')

os.chdir(DRIVE_DIR)
print(f'Working directory: {os.getcwd()}')

Updating d291cb5..b01d90f
Fast-forward
 experiments/colab_iterative_refinement.ipynb | 214 +-----------------------
 experiments/iterative_refinement.py          | 207 ++++++++++++++---------
 tests/test_iterative_config.py               | 239 +++++++++++++++++++++++++++
 3 files changed, 378 insertions(+), 282 deletions(-)
 create mode 100644 tests/test_iterative_config.py

Working directory: /content/drive/MyDrive/AC-Solver-Caltech


## Cell 3 — Install Dependencies

In [ ]:
import subprocess
result = subprocess.run(
    ['pip', 'install', '-q', '-r', 'requirements.txt'],
    cwd=DRIVE_DIR, capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else 'Done.')
if result.returncode != 0:
    print('ERRORS:', result.stderr[-1000:])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 96.0 MB/s eta 0:00:00



## Cell 4 — Verify Setup

In [ ]:
import os, json

checks = {
    'Original model checkpoint':  'value_search/checkpoints/best_mlp.pt',
    'Feature stats':              'value_search/checkpoints/feature_stats.json',
    'Greedy solved presentations': 'ac_solver/search/miller_schupp/data/greedy_solved_presentations.txt',
    'Greedy search paths':         'ac_solver/search/miller_schupp/data/greedy_search_paths.txt',
    'Config':                      'experiments/config.yaml',
}

all_ok = True
for label, path in checks.items():
    full = os.path.join(DRIVE_DIR, path)
    exists = os.path.exists(full)
    status = '✓' if exists else '✗  MISSING'
    print(f'  {status}  {label}')
    if not exists:
        all_ok = False

# Check refinement state (if resuming)
state_file = os.path.join(DRIVE_DIR, 'experiments/refinement/refinement_state.json')
if os.path.exists(state_file):
    with open(state_file) as f:
        state = json.load(f)
    print(f'\n  Saved state found: iteration={state["iteration"]}, '
          f'total_solved_history={state["total_solved_per_iteration"]}')
else:
    print('\n  No saved state — will start fresh.')

if all_ok:
    print('\nAll checks passed. Ready to run.')
else:
    print('\nFix missing files before running.')

  ✓  Original model checkpoint
  ✓  Feature stats
  ✓  Greedy solved presentations
  ✓  Greedy search paths
  ✓  Config

  Saved state found: iteration=2, total_solved_history=[533, 539, 539, 539, 539, 539, 556]

All checks passed. Ready to run.


## Cell 5 — Configuration
Edit these before running. Then run Cell 6 (fresh start) or Cell 8 (resume).

In [ ]:
# ── Tune these ────────────────────────────────────────────────────────────────
MAX_ITERATIONS      = 5       # How many search→train cycles
MAX_PATH_LENGTH     = 1200     # Reject training paths longer than this
MAX_NODES           = 1_000_000 # 100K nodes: ~25s per presentation, ~5hrs per iteration
TIME_LIMIT_PER_PRES = 90      # Seconds per presentation for search (45–90)

# ── Architecture / algorithm toggles ──────────────────────────────────────────
RUN_MLP     = False  # Include MLP (v-guided greedy with MLP model)
RUN_SEQ     = True   # Include Seq (v-guided greedy with seq model)
ENABLE_MCTS = False  # MCTS search  (very slow; keep False unless you have many hours)
ENABLE_BEAM = False  # Beam search  (k=10; finds solutions but paths are very long)
# ──────────────────────────────────────────────────────────────────────────────

import sys
sys.path.insert(0, DRIVE_DIR)

print('Config set:')
print(f'  max_iterations      = {MAX_ITERATIONS}')
print(f'  max_path_length     = {MAX_PATH_LENGTH}')
print(f'  max_nodes           = {MAX_NODES:,}')
print(f'  time_limit_per_pres = {TIME_LIMIT_PER_PRES}s')
print(f'  run_mlp             = {RUN_MLP}')
print(f'  run_seq             = {RUN_SEQ}')
print(f'  enable_mcts         = {ENABLE_MCTS}')
print(f'  enable_beam         = {ENABLE_BEAM}')


# ── Progress streaming helper (used by Cells 6 and 8) ─────────────────────────
def run_with_progress(cmd, cwd):
    """Run cmd as subprocess, printing headers normally and progress on one line."""
    import subprocess, sys, os

    HEADER_PATTERNS = [
        '===', 'ITERATION', '--- Step', 'complete:', 'REFINEMENT COMPLETE',
        'ERROR', 'Traceback', 'File "', 'Error:', 'Running:', 'Resuming:',
        'Total presentations', 'Greedy baseline', 'Resumed from',
        'No previous state', 'Loading greedy', 'Converged', 'Interrupted',
        'State saved', '[idx=', 'Search:',
    ]

    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'  # force line-by-line flushing in subprocess

    proc = subprocess.Popen(
        cmd, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env,
    )

    last_was_progress = False
    for raw in proc.stdout:
        line = raw.rstrip()
        if not line:
            continue
        is_header = any(p in line for p in HEADER_PATTERNS)
        if is_header:
            if last_was_progress:
                sys.stdout.write('\n')
            print(line)
            last_was_progress = False
        else:
            sys.stdout.write(f'\r  {line:<90}')
            sys.stdout.flush()
            last_was_progress = True

    if last_was_progress:
        sys.stdout.write('\n')
    proc.wait()
    return proc.returncode

Config set:
  max_iterations      = 5
  max_path_length     = 1200
  max_nodes           = 1,000,000
  time_limit_per_pres = 90s
  run_mlp             = False
  run_seq             = True
  enable_mcts         = False
  enable_beam         = False


## Cell 6 — Fresh Start
Run this to begin a new refinement run from the greedy baseline. **Skip if resuming an existing run.**

In [ ]:
cmd = [
    sys.executable,
    os.path.join(DRIVE_DIR, 'experiments/iterative_refinement.py'),
    '--max-iterations', str(MAX_ITERATIONS),
    '--max-path-length', str(MAX_PATH_LENGTH),
    '--time-limit-per-pres', str(TIME_LIMIT_PER_PRES),
]
if MAX_NODES is not None:
    cmd.extend(['--max-nodes', str(MAX_NODES)])
if RUN_MLP:
    cmd.append('--run-mlp')
if not RUN_SEQ:
    cmd.append('--no-seq')
if ENABLE_MCTS:
    cmd.append('--enable-mcts')
if ENABLE_BEAM:
    cmd.append('--enable-beam')

print('Running:', ' '.join(cmd))
rc = run_with_progress(cmd, cwd=DRIVE_DIR)
print('Exit code:', rc)

## Cell 7 — Reconstruct State (run only if state files are missing)
Run this **only** when `refinement_state.json` is missing but partial search results exist on Drive (e.g. after accidentally deleting state or after a very early crash before any state was saved).  
It auto-detects the latest partial results directory and rebuilds both state files.  
Then run Cell 8 to continue.

In [ ]:
import os, sys, json

sys.path.insert(0, DRIVE_DIR)
from value_search.data_extraction import load_presentations, load_paths

DATA_DIR       = os.path.join(DRIVE_DIR, 'ac_solver/search/miller_schupp/data')
REFINEMENT_DIR = os.path.join(DRIVE_DIR, 'experiments/refinement')
RESULTS_BASE   = os.path.join(DRIVE_DIR, 'experiments/results')
os.makedirs(REFINEMENT_DIR, exist_ok=True)

# 1. Auto-detect the latest partial results directory
candidate_dirs = sorted([
    d for d in os.listdir(RESULTS_BASE)
    if os.path.isdir(os.path.join(RESULTS_BASE, d))
    and os.path.exists(os.path.join(RESULTS_BASE, d, 'v_guided_greedy_progress.jsonl'))
])
if not candidate_dirs:
    print("ERROR: No partial results found in", RESULTS_BASE)
    raise SystemExit

PARTIAL_DIR = os.path.join(RESULTS_BASE, candidate_dirs[-1])
print(f"Using partial results dir: {PARTIAL_DIR}")

# Show what's in it
jsonl = os.path.join(PARTIAL_DIR, 'v_guided_greedy_progress.jsonl')
total, solved_indices = 0, []
with open(jsonl) as f:
    for line in f:
        r = json.loads(line)
        total += 1
        if r.get('solved'):
            solved_indices.append(r['idx'])
print(f"  Contains: {total} processed, {len(solved_indices)} solved: {solved_indices}")

# 2. Reload greedy paths from source data
solved_pres = load_presentations(os.path.join(DATA_DIR, 'greedy_solved_presentations.txt'))
raw_paths   = load_paths(os.path.join(DATA_DIR, 'greedy_search_paths.txt'))
greedy_paths = {
    tuple(pres): [[a - 1, l] for a, l in raw_path[1:]]
    for pres, raw_path in zip(solved_pres, raw_paths)
}
print(f"Greedy paths loaded: {len(greedy_paths)}")

# 3. Write all_solved_paths.json
paths_file = os.path.join(REFINEMENT_DIR, 'all_solved_paths.json')
with open(paths_file, 'w') as f:
    json.dump({str(list(k)): v for k, v in greedy_paths.items()}, f)
print(f"Written: all_solved_paths.json ({len(greedy_paths)} paths)")

# 4. Write refinement_state.json pointing at partial results
state = {
    "iteration": 0,
    "solved_per_iteration": [],
    "total_solved_per_iteration": [len(greedy_paths)],
    "model_paths": [],
    "results_dirs": [PARTIAL_DIR],
}
state_file = os.path.join(REFINEMENT_DIR, 'refinement_state.json')
with open(state_file, 'w') as f:
    json.dump(state, f, indent=2)
print(f"Written: refinement_state.json")
print(f"\nReady — run Cell 8 (Resume).")

Using partial results dir: /content/drive/MyDrive/AC-Solver-Caltech/experiments/results/2026-02-26_04-06-25
  Contains: 397 processed, 6 solved: [599, 689, 698, 711, 712, 834]
Greedy paths loaded: 533
Written: all_solved_paths.json (533 paths)
Written: refinement_state.json

Ready — run Cell 8 (Resume).


## Cell 8 — Resume After Disconnect
After Colab disconnects: re-run Cells 1 → 2 → 3 → 5, then run this cell.  
Picks up from the last completed iteration automatically — state persists on Drive.

In [ ]:
cmd = [
    sys.executable,
    os.path.join(DRIVE_DIR, 'experiments/iterative_refinement.py'),
    '--resume',
    '--max-iterations', str(MAX_ITERATIONS),
    '--max-path-length', str(MAX_PATH_LENGTH),
    '--time-limit-per-pres', str(TIME_LIMIT_PER_PRES),
]
if MAX_NODES is not None:
    cmd.extend(['--max-nodes', str(MAX_NODES)])
if RUN_MLP:
    cmd.append('--run-mlp')
if not RUN_SEQ:
    cmd.append('--no-seq')
if ENABLE_MCTS:
    cmd.append('--enable-mcts')
if ENABLE_BEAM:
    cmd.append('--enable-beam')

print('Resuming:', ' '.join(cmd))
rc = run_with_progress(cmd, cwd=DRIVE_DIR)
print('Exit code:', rc)

Resuming: /usr/bin/python3 /content/drive/MyDrive/AC-Solver-Caltech/experiments/iterative_refinement.py --resume --max-iterations 5 --max-path-length 1200 --time-limit-per-pres 90 --max-nodes 1000000
    AC-Solver Iterative Refinement Pipeline                                                 
    Time limit:     90s/presentation                                                        
  Search:         Seq
    State file:     /content/drive/MyDrive/AC-Solver-Caltech/experiments/refinement/refinement_state.json
  Total presentations: 1190
  Resumed from iteration 2
    Previously solved: 556/1190                                                             
  ITERATION 2
    Total solved so far: 556/1190                                                           
  Search:  Seq
    Searching 634/1190 unsolved presentations                                               
  --- Step 1: Seq Search (634 unsolved) ---
    Resuming seq search from: /content/drive/MyDrive/AC-Solver-Caltech/experime

## Cell 10 — Check Progress Anytime

In [ ]:
import os, json

state_file = os.path.join(DRIVE_DIR, 'experiments/refinement/refinement_state.json')
if not os.path.exists(state_file):
    print('No state file — run Cell 6 (fresh start) or Cell 7 (reconstruct state) first.')
else:
    with open(state_file) as f:
        state = json.load(f)

    paths_file = os.path.join(DRIVE_DIR, 'experiments/refinement/all_solved_paths.json')
    total_solved = 0
    if os.path.exists(paths_file):
        with open(paths_file) as f:
            total_solved = len(json.load(f))

    print(f'Current iteration:  {state["iteration"]}')
    print(f'Total solved:       {total_solved}/1190')
    print(f'Solved per iter:    {state["solved_per_iteration"]}')
    print(f'Total per iter:     {state["total_solved_per_iteration"]}')

    refinement_dir = os.path.join(DRIVE_DIR, 'experiments/refinement')
    print(f'\nFiles in refinement dir:')
    for f in sorted(os.listdir(refinement_dir)):
        fpath = os.path.join(refinement_dir, f)
        size_mb = os.path.getsize(fpath) / 1e6 if os.path.isfile(fpath) else 0
        tag = f'({size_mb:.1f} MB)' if os.path.isfile(fpath) else '(dir)'
        print(f'  {f}  {tag}')